In [ ]:
# CELL 1: IMPORTS
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors

from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

import warnings
warnings.filterwarnings('ignore')

print("✓ Imports done")

# CELL 2: LOAD DATA
df_full = pd.read_csv('../../../data/002_processed_data/xy_df_full.csv')
df_kaggle = pd.read_csv('../../../data/003_kaggle_competition_data/25261_regression_sample_submission.csv')
df_full = df_full.dropna(subset=['homo', 'lumo'])

descriptor_cols = [col for col in df_full.columns if col not in ['smiles', 'homo', 'lumo', 'stability']]
X = df_full[descriptor_cols].values
y_homo = df_full['homo'].values
y_lumo = df_full['lumo'].values

descriptor_names = [x[0] for x in Descriptors._descList]

def compute_descriptors(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return [np.nan] * len(descriptor_names)
    return [getattr(Descriptors, name)(mol) for name in descriptor_names]

X_kaggle = pd.DataFrame(
    [compute_descriptors(s) for s in df_kaggle['smiles'].tolist()], 
    columns=descriptor_names
)

var_sel = VarianceThreshold(threshold=0)
imp = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_clean = var_sel.fit_transform(X)
X_imp = imp.fit_transform(X_clean)
X_scaled = scaler.fit_transform(X_imp)

X_kaggle_clean = var_sel.transform(X_kaggle)
X_kaggle_imp = imp.transform(X_kaggle_clean)
X_kaggle_scaled = scaler.transform(X_kaggle_imp)

print(f"Train: {X_imp.shape}, Kaggle: {X_kaggle_imp.shape}")

# CELL 3: PRECOMPUTE ALL SAMPLE SCORES
print("\nPrecomputing sample scores...")

scores = {}

# GMM scores
for n_comp in [2, 3, 4, 5]:
    gmm = GaussianMixture(n_components=n_comp, random_state=42)
    gmm.fit(X_kaggle_scaled)
    scores[f'gmm_{n_comp}'] = gmm.score_samples(X_scaled)

# IsolationForest scores
for cont in [0.05, 0.1, 0.15, 0.2]:
    iso = IsolationForest(contamination=cont, random_state=42, n_estimators=100)
    iso.fit(X_kaggle_scaled)
    scores[f'iso_{cont}'] = iso.decision_function(X_scaled)

# LOF scores
for k in [10, 20, 30, 50]:
    lof = LocalOutlierFactor(n_neighbors=k, novelty=True)
    lof.fit(X_kaggle_scaled)
    scores[f'lof_{k}'] = lof.decision_function(X_scaled)

print(f"✓ Precomputed {len(scores)} score methods")

# CELL 4: EVALUATION FUNCTION
def evaluate(X_imp, y, score_key, n, k, alpha):
    sc = scores[score_key]
    idx = np.argsort(sc)[-n:]  # top n most similar to Kaggle
    X_sub = X_imp[idx]
    y_sub = y[idx]
    
    sel = SelectKBest(f_regression, k=k)
    X_sel = sel.fit_transform(X_sub, y_sub)
    
    model = Ridge(alpha=alpha)
    cv = cross_val_score(model, X_sel, y_sub, cv=5, scoring='neg_mean_absolute_error')
    return -cv.mean()

# CELL 5: GRID SEARCH ALL METHODS
print("\n" + "="*60)
print("HOMO - Testing all methods")
print("="*60)

best_homo = {'mae': 999}

for score_key in scores.keys():
    for n in [70, 80, 90, 100, 120, 150]:
        for k in [30, 40, 50, 60]:
            if k >= n - 10:
                continue
            for alpha in [0.1, 0.3, 0.5, 1.0, 2.0, 5.0]:
                try:
                    mae = evaluate(X_imp, y_homo, score_key, n, k, alpha)
                    if mae < best_homo['mae']:
                        best_homo = {'mae': mae, 'method': score_key, 'n': n, 'k': k, 'alpha': alpha}
                        print(f"  New best: {score_key}, n={n}, k={k}, α={alpha}, MAE={mae:.4f}")
                except:
                    pass

print(f"\n✓ Best HOMO: {best_homo}")

print("\n" + "="*60)
print("LUMO - Testing all methods")
print("="*60)

best_lumo = {'mae': 999}

for score_key in scores.keys():
    for n in [70, 80, 90, 100, 120, 150]:
        for k in [30, 40, 50, 60]:
            if k >= n - 10:
                continue
            for alpha in [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]:
                try:
                    mae = evaluate(X_imp, y_lumo, score_key, n, k, alpha)
                    if mae < best_lumo['mae']:
                        best_lumo = {'mae': mae, 'method': score_key, 'n': n, 'k': k, 'alpha': alpha}
                        print(f"  New best: {score_key}, n={n}, k={k}, α={alpha}, MAE={mae:.4f}")
                except:
                    pass

print(f"\n✓ Best LUMO: {best_lumo}")

# CELL 6: RESULTS AND SUBMISSION
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)

avg = (best_homo['mae'] + best_lumo['mae']) / 2
print(f"\nHOMO: MAE={best_homo['mae']:.4f}")
print(f"  Method: {best_homo['method']}, n={best_homo['n']}, k={best_homo['k']}, α={best_homo['alpha']}")
print(f"\nLUMO: MAE={best_lumo['mae']:.4f}")
print(f"  Method: {best_lumo['method']}, n={best_lumo['n']}, k={best_lumo['k']}, α={best_lumo['alpha']}")
print(f"\nAverage MAE: {avg:.4f}")

# Build final models
sc_h = scores[best_homo['method']]
idx_h = np.argsort(sc_h)[-best_homo['n']:]
X_h = X_imp[idx_h]
y_h = y_homo[idx_h]

sel_h = SelectKBest(f_regression, k=best_homo['k'])
X_h_sel = sel_h.fit_transform(X_h, y_h)
model_h = Ridge(alpha=best_homo['alpha'])
model_h.fit(X_h_sel, y_h)

sc_l = scores[best_lumo['method']]
idx_l = np.argsort(sc_l)[-best_lumo['n']:]
X_l = X_imp[idx_l]
y_l = y_lumo[idx_l]

sel_l = SelectKBest(f_regression, k=best_lumo['k'])
X_l_sel = sel_l.fit_transform(X_l, y_l)
model_l = Ridge(alpha=best_lumo['alpha'])
model_l.fit(X_l_sel, y_l)

# Predict
pred_homo = model_h.predict(sel_h.transform(X_kaggle_imp))
pred_lumo = model_l.predict(sel_l.transform(X_kaggle_imp))

pred_homo = np.clip(pred_homo, y_homo.min(), y_homo.max())
pred_lumo = np.clip(pred_lumo, y_lumo.min(), y_lumo.max())

print(f"\nPredictions:")
print(f"  HOMO: mean={pred_homo.mean():.4f}, std={pred_homo.std():.4f}")
print(f"  LUMO: mean={pred_lumo.mean():.4f}, std={pred_lumo.std():.4f}")

# Save
submission = pd.DataFrame({
    'smiles': df_kaggle['smiles'],
    'homo': pred_homo,
    'lumo': pred_lumo
})
submission.to_csv('../../../data/003_kaggle_competition_data/25261_regression_sample_submission.csv', index=False)
print(f"\n✓ Saved submission")

In [ ]:
# CELL 7: FEATURE IMPORTANCE ANALYSIS
print("\n" + "="*60)
print("TOP SELECTED FEATURES - REGRESSION")
print("="*60)

# Get feature names that passed variance threshold
var_mask = var_sel.get_support()
features_after_var = [descriptor_cols[i] for i in range(len(descriptor_cols)) if var_mask[i]]

# HOMO features
print("\n--- HOMO Model (LOF, k=60) ---")
selected_mask_h = sel_h.get_support()
selected_indices_h = sel_h.get_support(indices=True)

homo_features = []
for idx, coef in zip(selected_indices_h, model_h.coef_):
    homo_features.append({
        'feature': features_after_var[idx],
        'coef': abs(coef)
    })

homo_features = sorted(homo_features, key=lambda x: x['coef'], reverse=True)

print(f"\nTop 10 features by coefficient magnitude:")
for i, feat in enumerate(homo_features[:10], 1):
    print(f"  {i:2d}. {feat['feature']:30s} (|coef|={feat['coef']:.4f})")

# LUMO features
print("\n--- LUMO Model (GMM, k=50) ---")
selected_mask_l = sel_l.get_support()
selected_indices_l = sel_l.get_support(indices=True)

lumo_features = []
for idx, coef in zip(selected_indices_l, model_l.coef_):
    lumo_features.append({
        'feature': features_after_var[idx],
        'coef': abs(coef)
    })

lumo_features = sorted(lumo_features, key=lambda x: x['coef'], reverse=True)

print(f"\nTop 10 features by coefficient magnitude:")
for i, feat in enumerate(lumo_features[:10], 1):
    print(f"  {i:2d}. {feat['feature']:30s} (|coef|={feat['coef']:.4f})")

# Summary
print("\n" + "="*60)
print("FEATURE OVERLAP ANALYSIS")
print("="*60)
homo_top10 = set([f['feature'] for f in homo_features[:10]])
lumo_top10 = set([f['feature'] for f in lumo_features[:10]])
overlap = homo_top10 & lumo_top10

print(f"Features in both HOMO and LUMO top 10: {len(overlap)}")
if overlap:
    for feat in overlap:
        print(f"  - {feat}")

In [ ]:
# CELL 8: VISUALIZATION SETUP
import matplotlib.pyplot as plt
import os

os.makedirs('../../../figures', exist_ok=True)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("✓ Visualization setup ready")

# CELL 9: LOAD DATASETS FOR COMPARISON
df_hopv = pd.read_csv('../../../data/002_processed_data/xy_df_hopv.csv')
df_sql = pd.read_csv('../../../../introduction_to_sql/sql_dataset.csv')
df_hopv = df_hopv.dropna(subset=['stability'])
df_sql = df_sql.dropna(subset=['stability'])

print(f"HOPV: {len(df_hopv)} samples")
print(f"SQL: {len(df_sql)} samples")

# CELL 10: CLASS DISTRIBUTION COMPARISON (HOPV vs SQL)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

class_order = ['high', 'low', 'medium']
colors = ['#2ecc71', '#e74c3c', '#f39c12']

# HOPV
hopv_counts = df_hopv['stability'].value_counts()
hopv_vals = [hopv_counts.get(c, 0) for c in class_order]
axes[0].bar(class_order, hopv_vals, color=colors, edgecolor='black', linewidth=1.2)
axes[0].set_title('HOPV Dataset (Used for Classification)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Stability Class')
axes[0].set_ylabel('Number of Molecules')
for i, val in enumerate(hopv_vals):
    axes[0].text(i, val + 2, str(val), ha='center', fontsize=12, fontweight='bold')

# SQL
sql_counts = df_sql['stability'].value_counts()
sql_vals = [sql_counts.get(c, 0) for c in class_order]
axes[1].bar(class_order, sql_vals, color=colors, edgecolor='black', linewidth=1.2)
axes[1].set_title('SQL Dataset (Excluded from Classification)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Stability Class')
axes[1].set_ylabel('Number of Molecules')
for i, val in enumerate(sql_vals):
    axes[1].text(i, val + 2, str(val), ha='center', fontsize=12, fontweight='bold')

plt.suptitle('Class Distribution Comparison: Why HOPV-Only Performed Better', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../../../figures/class_distribution_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: class_distribution_comparison.png")

# CELL 11: HOMO/LUMO DISTRIBUTION - TRAINING DATA
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# HOMO distribution
axes[0].hist(y_homo, bins=30, color='#3498db', edgecolor='black', linewidth=0.8, alpha=0.8)
axes[0].axvline(y_homo.mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean: {y_homo.mean():.2f} eV')
axes[0].set_xlabel('HOMO Energy (eV)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('HOMO Energy Distribution (Training Data)', fontsize=14, fontweight='bold')
axes[0].legend()

# LUMO distribution
axes[1].hist(y_lumo, bins=30, color='#9b59b6', edgecolor='black', linewidth=0.8, alpha=0.8)
axes[1].axvline(y_lumo.mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean: {y_lumo.mean():.2f} eV')
axes[1].set_xlabel('LUMO Energy (eV)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('LUMO Energy Distribution (Training Data)', fontsize=14, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../../../figures/homo_lumo_distribution_train.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: homo_lumo_distribution_train.png")

# CELL 12: HOMO/LUMO DISTRIBUTION - KAGGLE PREDICTIONS
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# HOMO predictions
axes[0].hist(pred_homo, bins=30, color='#3498db', edgecolor='black', linewidth=0.8, alpha=0.8)
axes[0].axvline(pred_homo.mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean: {pred_homo.mean():.2f} eV')
axes[0].set_xlabel('HOMO Energy (eV)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('HOMO Energy Distribution (Kaggle Predictions)', fontsize=14, fontweight='bold')
axes[0].legend()

# LUMO predictions
axes[1].hist(pred_lumo, bins=30, color='#9b59b6', edgecolor='black', linewidth=0.8, alpha=0.8)
axes[1].axvline(pred_lumo.mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean: {pred_lumo.mean():.2f} eV')
axes[1].set_xlabel('LUMO Energy (eV)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('LUMO Energy Distribution (Kaggle Predictions)', fontsize=14, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../../../figures/homo_lumo_distribution_kaggle.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: homo_lumo_distribution_kaggle.png")

# CELL 13: ACTUAL VS PREDICTED (CV using selected samples)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# HOMO: Actual vs Predicted on selected training samples
y_h_pred_train = model_h.predict(X_h_sel)
axes[0].scatter(y_h, y_h_pred_train, alpha=0.6, color='#3498db', edgecolor='black', linewidth=0.5)
min_val, max_val = min(y_h.min(), y_h_pred_train.min()), max(y_h.max(), y_h_pred_train.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual HOMO (eV)')
axes[0].set_ylabel('Predicted HOMO (eV)')
axes[0].set_title(f'HOMO: Actual vs Predicted\n(n={best_homo["n"]} samples, MAE={best_homo["mae"]:.4f})', fontsize=12, fontweight='bold')
axes[0].legend()

# LUMO: Actual vs Predicted on selected training samples
y_l_pred_train = model_l.predict(X_l_sel)
axes[1].scatter(y_l, y_l_pred_train, alpha=0.6, color='#9b59b6', edgecolor='black', linewidth=0.5)
min_val, max_val = min(y_l.min(), y_l_pred_train.min()), max(y_l.max(), y_l_pred_train.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual LUMO (eV)')
axes[1].set_ylabel('Predicted LUMO (eV)')
axes[1].set_title(f'LUMO: Actual vs Predicted\n(n={best_lumo["n"]} samples, MAE={best_lumo["mae"]:.4f})', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../../../figures/actual_vs_predicted_regression.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: actual_vs_predicted_regression.png")

# CELL 14: RESIDUAL DISTRIBUTION
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# HOMO residuals
residuals_h = y_h - y_h_pred_train
axes[0].hist(residuals_h, bins=25, color='#3498db', edgecolor='black', linewidth=0.8, alpha=0.8)
axes[0].axvline(0, color='#e74c3c', linestyle='--', linewidth=2)
axes[0].set_xlabel('Residual (Actual - Predicted) eV')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'HOMO Residual Distribution\nMean: {residuals_h.mean():.4f}, Std: {residuals_h.std():.4f}', fontsize=12, fontweight='bold')

# LUMO residuals
residuals_l = y_l - y_l_pred_train
axes[1].hist(residuals_l, bins=25, color='#9b59b6', edgecolor='black', linewidth=0.8, alpha=0.8)
axes[1].axvline(0, color='#e74c3c', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual (Actual - Predicted) eV')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'LUMO Residual Distribution\nMean: {residuals_l.mean():.4f}, Std: {residuals_l.std():.4f}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../../../figures/residual_distribution_regression.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: residual_distribution_regression.png")

# CELL 15: TOP FEATURES BAR CHARTS (HOMO and LUMO)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# HOMO features
top_n = 10
homo_names = [f['feature'] for f in homo_features[:top_n]]
homo_coefs = [f['coef'] for f in homo_features[:top_n]]

axes[0].barh(range(len(homo_names)), homo_coefs, color='#3498db', edgecolor='black', linewidth=0.8)
axes[0].set_yticks(range(len(homo_names)))
axes[0].set_yticklabels(homo_names)
axes[0].invert_yaxis()
axes[0].set_xlabel('|Coefficient|')
axes[0].set_title('HOMO: Top 10 Features by Coefficient Magnitude', fontsize=12, fontweight='bold')
for i, coef in enumerate(homo_coefs):
    axes[0].text(coef + 0.002, i, f'{coef:.3f}', va='center', fontsize=9)

# LUMO features
lumo_names = [f['feature'] for f in lumo_features[:top_n]]
lumo_coefs = [f['coef'] for f in lumo_features[:top_n]]

axes[1].barh(range(len(lumo_names)), lumo_coefs, color='#9b59b6', edgecolor='black', linewidth=0.8)
axes[1].set_yticks(range(len(lumo_names)))
axes[1].set_yticklabels(lumo_names)
axes[1].invert_yaxis()
axes[1].set_xlabel('|Coefficient|')
axes[1].set_title('LUMO: Top 10 Features by Coefficient Magnitude', fontsize=12, fontweight='bold')
for i, coef in enumerate(lumo_coefs):
    axes[1].text(coef + 0.002, i, f'{coef:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../../../figures/top_features_regression.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: top_features_regression.png")

# CELL 16: BAND GAP DISTRIBUTION (LUMO - HOMO)
fig, ax = plt.subplots(figsize=(10, 5))

# Training band gap
bandgap_train = y_lumo - y_homo
# Kaggle predicted band gap
bandgap_kaggle = pred_lumo - pred_homo

ax.hist(bandgap_train, bins=25, color='#3498db', edgecolor='black', linewidth=0.8, alpha=0.6, label=f'Training (mean={bandgap_train.mean():.2f} eV)')
ax.hist(bandgap_kaggle, bins=25, color='#e74c3c', edgecolor='black', linewidth=0.8, alpha=0.6, label=f'Kaggle Pred (mean={bandgap_kaggle.mean():.2f} eV)')

# Optimal band gap range
ax.axvspan(1.1, 1.5, alpha=0.2, color='green', label='Optimal Range (1.1-1.5 eV)')

ax.set_xlabel('Band Gap (LUMO - HOMO) eV')
ax.set_ylabel('Frequency')
ax.set_title('Band Gap Distribution: Training vs Kaggle Predictions', fontsize=14, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('../../../figures/bandgap_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: bandgap_distribution.png")

# CELL 17: VISUALIZATION SUMMARY
print("="*60)
print("REGRESSION VISUALIZATION SUMMARY")
print("="*60)
print("\nGenerated figures:")
print("  1. class_distribution_comparison.png - HOPV vs SQL stability classes")
print("  2. homo_lumo_distribution_train.png - Training data energy distributions")
print("  3. homo_lumo_distribution_kaggle.png - Kaggle prediction distributions")
print("  4. actual_vs_predicted_regression.png - Model fit quality")
print("  5. residual_distribution_regression.png - Prediction errors")
print("  6. top_features_regression.png - Feature importance (HOMO & LUMO)")
print("  7. bandgap_distribution.png - Band gap analysis")
print("\nAll figures saved to: ../../../figures/")
print("="*60)